In [ ]:
from torchvision import models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim


**Model banana**

In [ ]:
model = models.vgg19(pretrained=True)
model

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padd

In [ ]:
for param in model.features.parameters():
  param.requires_grad=False #imagenet weights fixed/frozen. freezes everything first

#unfreezing last block, for improvement
for param in model.features[28:].parameters():
    param.requires_grad = True

In [ ]:
#only change/replace classifier
model.classifier[6]= nn.Linear(4096, 4)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
model = model.to(device)

cuda


**Dataset and DataLoader creation**

In [ ]:
import zipfile

with zipfile.ZipFile("data.zip", "r") as zip_ref:
    zip_ref.extractall("data")

In [ ]:
data = "/content/data"

In [ ]:
#seperate transforms for train and test bcs we shouldnt augment(flip, rotation) test data.
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [ ]:
train_dataset = datasets.ImageFolder(
    root = 'data/data/Training',
    transform=train_transform
)

test_dataset = datasets.ImageFolder(
    root = 'data/data/Testing',
    transform=test_transform
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    num_workers=2,
    pin_memory=True,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    num_workers=2,
    pin_memory=True,
    shuffle=False
)

**create Loss and optimizer**

In [ ]:
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-5) #note only optimizing classifier


**Training Loop**

In [ ]:
epochs = 20
current_loss =0

for epoch in range(epochs):
  current_loss = 0
  for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)

    outputs = model(images)
    loss = criteria(outputs, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    current_loss += loss.item()

  print(f"Epoch {epoch+1}, Loss:{current_loss/len(train_loader): .4f}")

Epoch 1, Loss: 0.3090
Epoch 2, Loss: 0.2335
Epoch 3, Loss: 0.1803
Epoch 4, Loss: 0.1417
Epoch 5, Loss: 0.1117
Epoch 6, Loss: 0.0862
Epoch 7, Loss: 0.0766
Epoch 8, Loss: 0.0553
Epoch 9, Loss: 0.0496
Epoch 10, Loss: 0.0447
Epoch 11, Loss: 0.0443
Epoch 12, Loss: 0.0304
Epoch 13, Loss: 0.0282
Epoch 14, Loss: 0.0273
Epoch 15, Loss: 0.0251
Epoch 16, Loss: 0.0205
Epoch 17, Loss: 0.0211
Epoch 18, Loss: 0.0201
Epoch 19, Loss: 0.0184
Epoch 20, Loss: 0.0166


**Testing Loop**

In [ ]:
##test accuracy
correct = 0
total = 0

model.eval() # eval mode

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1) # get class with highest score

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

model.train() # switch back to training mode if more training follows


Test Accuracy: 93.25%


VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padd